# T10: rich dumps through the stage cache (block `train/checks/stage_cache.py`)

Same pipeline and settings as the T6 `Rich_Dump_Run` (27 queries or a subset, K = 50, M = 10, LoFTR-MINIMA), but every query's canvas, the
whole K = 50 coarse shortlist and DINO's score of each place are cached in `CACHE`, and the M places that go to fine matching come from
`SELECT` (`dino`: the pipeline's order; `mix_<d>_<c>`: first d of the DINO order + first c of the coarse order not already in).
Env: GATE_DATA, POOL_PHOTOS, PHOTOS_FILE or (NUDT_SEED, N_NUDT, FRESH_EXCLUDE_POOL, EXCLUDE_IDS_FILE), TAG, CACHE, SELECT, QUERIES
(`all` / `std`), SHARE_FFT (1 = reuse the map-window FFTs between queries), N_LIMIT.

In [1]:
import os, sys, time
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
DATA = os.environ.get("GATE_DATA", "official"); POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))
TAG = os.environ["TAG"]; CACHE = os.environ.get("CACHE"); SELECT = os.environ.get("SELECT", "dino"); QUERIES = os.environ.get("QUERIES", "all")
SHARE_FFT = os.environ.get("SHARE_FFT", "0") == "1"; N_LIMIT = int(os.environ.get("N_LIMIT", "0"))
OUT = ROOT / "tests/Test_Iteration_10/results/rich_dump" / TAG; OUT.mkdir(parents=True, exist_ok=True)
from train.checks.stage_cache import CachedStagePipeline, select_dino, select_mixture, share_map_fft
if SHARE_FFT: share_map_fft()
if SELECT == "dino": select, M = None, 10
else:
    d, c = (int(v) for v in SELECT.split("_")[1:]); select, M = select_mixture(d, c), d + c
headings = [-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0] if QUERIES == "all" else [-5.0, 0.0, 5.0]
scales = [0.95, 1.0, 1.05] if QUERIES == "all" else [1.0, 1.05]
print(DATA, TAG, CACHE, SELECT, QUERIES, "share_fft", SHARE_FFT)

official cache_reuse /home/palash/GeoLoc/tests/Test_Iteration_10/results/cache/official dino all share_fft False


In [2]:
from train.checks.rich_dump import pack_photo
pipeline = CachedStagePipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=M,
                               matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=0.22, satellite="new",
                               map_margin_m=None, loftr_coarse_threshold=0.21, dino_weight=None, headings=headings, scale_factors=scales,
                               cache_dir=CACHE, select=select)
t0 = time.time(); pipeline.setup(POOL_PHOTOS, 7); print("setup %.0f s" % (time.time() - t0))
test_set = pipeline.test_set; n_all = len(test_set.items)
if os.environ.get("PHOTOS_FILE"):
    indices = sorted(int(i) for i in pd.read_csv(os.environ["PHOTOS_FILE"]).photo)
else:                                                         # a fresh NUDT draw, as in the T6 dump notebook
    from train.datasets import nudt_data
    seen = set(int(i) for i in nudt_data.spread_sample(test_set.items, int(os.environ["FRESH_EXCLUDE_POOL"]), 7))
    seen |= set(int(i) for i in pd.read_csv(os.environ["EXCLUDE_IDS_FILE"]).photo)
    pool = np.array([i for i in range(n_all) if i not in seen])
    indices = sorted(int(i) for i in np.random.default_rng(int(os.environ["NUDT_SEED"])).choice(pool, int(os.environ["N_NUDT"]), replace=False))
if N_LIMIT: indices = indices[:N_LIMIT]
pd.DataFrame(dict(photo=indices)).to_csv(OUT / "photos.csv", index=False)
len(indices)

level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 99 s


/home/palash/Ideaforge/GeoLoc/.venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/palash/Ideaforge/GeoLoc/.venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup 112 s


5

In [3]:
import torch
done = {int(p.stem.split("_")[1]) for p in OUT.glob("photo_*.npz")}
start = time.time(); failures = []; secs = []
for count, index in enumerate([i for i in indices if i not in done]):
    t1 = time.time()
    try:
        found = pipeline.run_queries(test_set.items[index], test_set.priors[index])
        arrays = pack_photo(test_set.items[index], test_set.truths[index], test_set.sigmas[index], found["level"], found["queries"], min_keep_confidence=0.15)
        np.savez_compressed(OUT / ("photo_%d.npz" % index), **arrays)
    except Exception as error:
        failures.append((index, repr(error)[:200])); print("photo", index, "failed", repr(error)[:200])
    secs.append(time.time() - t1)
    if count % 20 == 0: print("photo %d | %.1f s per photo" % (count, np.mean(secs)), flush=True)
pd.DataFrame(failures, columns=["photo", "error"]).to_csv(OUT / "failures.csv", index=False)
pd.DataFrame(dict(seconds=secs)).to_csv(OUT / "seconds.csv", index=False)
dict(photos=len(list(OUT.glob("photo_*.npz"))), failed=len(failures), mean_s=round(float(np.mean(secs)) if secs else 0, 2), total_s=round(time.time() - start))

photo 0 | 4.9 s per photo


{'photos': 5, 'failed': 0, 'mean_s': 4.7, 'total_s': 23}